# Classical CV pipeline - per-larva tracker (representative notebook)

This notebook is the **representative example** of the classical computer-vision
tracking stage used in the study. In the original work a separate, structurally
identical notebook was used for each larva and condition, because the masking,
threshold and morphology parameters were tuned manually per dish/larva. All those
notebooks share the same code; only the parameters differ. This single notebook
documents that code.

Input:  `../records/<dataset>/isolated_dishes/<condition>_dish_<larva>.mp4`
        (produced by `02_dish_isolation.ipynb`)
Output: `tracking/<dataset>/<condition>/larva_positions_<larva>.csv`

Sections:
1. Imports
2. Larva tracking - algorithm
3. Missing-value computation
4. Movement trajectory (mm & px)
5. Reverse engineering (path from angles & distances)
6. Heatmap (most visited positions)
7. Bubble plot (most visited positions)
8. Rose diagram (movement directions)


## 1. Imports

In [ ]:
import cv2 as cv
import time
import numpy as np
import csv
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from typing import List
import pandas as pd
import os

In [ ]:
def close_cv_windows():
    """Emergency, reliable closing of all OpenCV windows.

    Run this cell when, after pressing 'q', the windows did not close and the
    kernel appears to hang. Repeated waitKey() forces the GUI event queue (Cocoa
    on macOS) to be processed and the windows to actually close - so the kernel
    does not need to be restarted.
    """
    for _ in range(5):
        cv.destroyAllWindows()
        cv.waitKey(1)
    print("OpenCV windows closed.")


# close_cv_windows()  # uncomment and run if the kernel hangs after 'q'

In [ ]:
# Maps a condition number to a file name.
# Names must match the files in the isolated_dishes directories exactly.
STUDY_TYPES = {
    1: "control",
    2: "pbs",
    3: "coli_2x10_8",
    4: "coli_5x10_7",
    5: "coli_5x10_8",
}

# Maps a dataset number to a recordings directory.
DATASETS = {
    1: "session1",
    2: "session2",
}

## 2. Larva tracking - algorithm

In [ ]:
def display_datasets():
    """Print the available datasets (recording directories)."""
    print("\n=== Available datasets ===")
    for num, name in DATASETS.items():
        print(f"{num}. {name}")
    print("==========================\n")

def get_dataset():
    """Read the dataset number from the user."""
    display_datasets()
    while True:
        try:
            dataset_num = int(input(f'Enter dataset number (1-{len(DATASETS)}): '))
            if dataset_num in DATASETS:
                return DATASETS[dataset_num]
            else:
                print(f'Error: choose a number between 1 and {len(DATASETS)}')
        except ValueError:
            print('Error: enter an integer')

def display_study_types():
    """Print the available study conditions."""
    print("\n=== Available conditions ===")
    for num, name in STUDY_TYPES.items():
        print(f"{num}. {name}")
    print("============================\n")

def get_study_type():
    """Read the condition number from the user."""
    display_study_types()
    while True:
        try:
            study_num = int(input('Enter condition number: '))
            if study_num in STUDY_TYPES:
                return STUDY_TYPES[study_num]
            else:
                print(f'Error: choose a number between 1 and {len(STUDY_TYPES)}')
        except ValueError:
            print('Error: enter an integer')

def get_larva_number():
    """Read the larva number from the user."""
    while True:
        try:
            larva_num = int(input('Enter larva number (1-6): '))
            if 1 <= larva_num <= 6:
                return larva_num
            else:
                print('Error: larva number must be between 1 and 6')
        except ValueError:
            print('Error: enter an integer')

### Example: larva 6

In [ ]:
class larvaTracker:
    def __init__(self, larva_number, study_type, dataset):
        self.larva_number = larva_number
        self.study_type = study_type  # store the condition in the object
        self.dataset = dataset  # recordings directory (e.g. session1)
        # File naming: {study_type}_dish_{larva_number}.mp4 in the chosen dataset
        self.video_path = f'../records/{dataset}/isolated_dishes/{study_type}_dish_{larva_number}.mp4'
        self.positions = []
        self.frame_number = 0
        self.fps = None
        self.frame_delay = None
        self.output_directory = Path('tracking')

    def open_video(self):
        capture = cv.VideoCapture(self.video_path)
        if not capture.isOpened():
            raise FileNotFoundError("Error opening the video file. File not found or wrong codec used!")
        self.fps = capture.get(cv.CAP_PROP_FPS)
        self.frame_delay = 1 / self.fps
        return capture

    @staticmethod
    def create_custom_margin_mask_with_rounded_corners(shape, left_margin, right_margin, top_margin, bottom_margin, corner_radius):
        mask = np.zeros(shape, dtype='uint8')
        height, width = shape
        central_rect = np.zeros(shape, dtype='uint8')
        cv.rectangle(central_rect, (left_margin + corner_radius, top_margin),
                     (width - right_margin - corner_radius, height - bottom_margin), 255, -1)
        cv.rectangle(central_rect, (left_margin, top_margin + corner_radius),
                     (width - right_margin, height - bottom_margin - corner_radius), 255, -1)
        mask = cv.bitwise_or(mask, central_rect)
        cv.circle(mask, (left_margin + corner_radius, top_margin + corner_radius), corner_radius, 255, -1)
        cv.circle(mask, (width - right_margin - corner_radius, top_margin + corner_radius), corner_radius, 255, -1)
        cv.circle(mask, (left_margin + corner_radius, height - bottom_margin - corner_radius), corner_radius, 255, -1)
        cv.circle(mask, (width - right_margin - corner_radius, height - bottom_margin - corner_radius), corner_radius, 255, -1)
        return mask

    def process_video(self):
        capture = self.open_video()
        # Tune the masking values here (per dish/larva).
        left_margin, right_margin, top_margin, bottom_margin, corner_radius = 20, 15, 20, 20, 15
        # Example per-larva parameters used in the study (PBS condition):
        # larva 1: margins = 48, 19, 18, 15, radius 40, thresh 40
        # larva 2: margins = 30, 19, 18, 15, radius 40, thresh 40
        # larva 3: margins = 20, 25, 18, 10, radius 40, thresh 50
        # larva 4: margins = 30, 25, 15, 10, radius 40, thresh 50
        # larva 5: margins = 30, 25, 10, 10, radius 40, thresh 50
        # larva 6: margins = 30, 19, 18,  5, radius 40
        scale = 0.2  # mm/px
        # Use the Petri-dish size as the canvas for drawn contours/centroids.
        plate_size_mm = 95
        plate_size_px = int(plate_size_mm / scale)
        previous_position = None

        while capture.isOpened():
            start_time = time.time()
            ret, frame = capture.read()
            if not ret:
                break

            # Preprocessing
            blank = np.zeros((plate_size_px, plate_size_px, 3), dtype='uint8')
            mask = self.create_custom_margin_mask_with_rounded_corners(frame.shape[:2], left_margin, right_margin, top_margin, bottom_margin, corner_radius)
            gray_frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
            masked_frame = cv.bitwise_and(gray_frame, gray_frame, mask=mask)
            # Blurring
            blur_frame = cv.GaussianBlur(masked_frame, (5, 5), cv.BORDER_REPLICATE)
            # Thresholding -> KEY PARAMETER "thresh" (TUNE AS NEEDED)
            _, thresh_frame = cv.threshold(src=blur_frame, thresh=40, maxval=255, type=cv.THRESH_BINARY_INV)
            # Opening (erosion -> dilation)
            kernel = np.ones((3, 3), np.uint8)
            eroded_frame = cv.erode(thresh_frame, kernel, iterations=3)
            dilated_frame = cv.dilate(eroded_frame, kernel, iterations=6)
            # Canny edge detection
            canny_frame = cv.Canny(dilated_frame, 100, 200)
            contours, _ = cv.findContours(canny_frame, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
            # Draw contours and centroids
            for contour in contours:
                area = cv.contourArea(contour)
                if 400 < area < 10000:
                    M = cv.moments(contour)
                    if M["m00"] != 0:
                        cx = int(M["m10"] / M["m00"])
                        cy = int(M["m01"] / M["m00"])
                        if previous_position:
                            distance = np.sqrt((cx - previous_position[0]) ** 2 + (cy - previous_position[1]) ** 2)
                        self.positions.append((self.frame_number, cx, cy))
                        previous_position = (cx, cy)
                        cv.circle(blank, (cx, cy), 5, (255, 0, 0), -1)
                    cv.drawContours(blank, [contour], -1, (0, 0, 255), 2)

            # Increment the frame number (needed when saving positions)
            self.frame_number += 1

            cv.imshow('Masked', masked_frame)
            cv.imshow('Thresh', thresh_frame)
            cv.imshow('Dilated', dilated_frame)
            # cv.imshow(f'Contours Larva {self.larva_number}', blank)

            if cv.waitKey(1) & 0xFF == ord('q'):
                break

            elapsed_time = time.time() - start_time
            time.sleep(max(0, self.frame_delay - elapsed_time))

        capture.release()
        # Reliable OpenCV window closing on macOS: a single waitKey(1) does not
        # close Cocoa windows and hangs the Jupyter kernel. The GUI event queue
        # must be pumped several times after destroyAllWindows().
        cv.destroyAllWindows()
        for _ in range(5):
            cv.waitKey(1)
        # self.save_positions()

    def save_positions(self):
        # Create tracking/<dataset>/<study_type> if it does not exist.
        study_dir = self.output_directory / str(self.dataset) / str(self.study_type)
        study_dir.mkdir(parents=True, exist_ok=True)

        filename = study_dir / f'larva_positions_{self.larva_number}.csv'
        with open(filename, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['Frame', 'X', 'Y'])
            for position in self.positions:
                writer.writerow(position)
        print(f'Positions saved to: {filename}')


# Run the tracker
dataset = get_dataset()
study_type = get_study_type()
larva_number = get_larva_number()

try:
    video_path = f'../records/{dataset}/isolated_dishes/{study_type}_dish_{larva_number}.mp4'
    if not os.path.isfile(video_path):
        raise FileNotFoundError(f"Video file does not exist: {video_path}")
    tracker = larvaTracker(larva_number, study_type, dataset)
    tracker.process_video()
except FileNotFoundError as e:
    print(e)
except ValueError:
    print("Invalid input. Make sure you enter an integer.")

## 3. Missing-value computation

### Total number of frames

In [ ]:
dataset = get_dataset()
study_type = get_study_type()
larva_number = get_larva_number()

In [ ]:
# Use dataset, study_type and larva_number from the previous cell
video_path = f'../records/{dataset}/isolated_dishes/{study_type}_dish_{larva_number}.mp4'
capture = cv.VideoCapture(video_path)

fps = capture.get(cv.CAP_PROP_FPS)
total_frames = int(capture.get(cv.CAP_PROP_FRAME_COUNT))

print(f'FPS: {fps}')
print(f'Total number of frames: {total_frames}')
print(f'Duration (minutes): {(total_frames//fps)/60:.2f}')

### Number of tracked frames

In [ ]:
tracked_frames = 0

# Use dataset, study_type and larva_number from the previous cell
file_path = f'tracking/{dataset}/{study_type}/larva_positions_{larva_number}.csv'

with open(file=file_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    next(reader)  # skip the header
    for row in reader:
        tracked_frames += 1

print(f'Number of tracked frames: {tracked_frames}')

### Missing-value computation

In [ ]:
missing_frames = total_frames - tracked_frames
missing_percentage = (missing_frames / total_frames) * 100
display(HTML(f'<h4>Missing frames: {missing_frames}'))
display(HTML(f'<h4>Missing data: <span style="color:red;">{missing_percentage:.2f} %'))

## 4. Movement trajectory (mm & px)

In [ ]:
class LarvaeTrajectoryPlotter:
    def __init__(self, file_path: str, larva_number: int, study_type: str, dataset: str):
        self.file_path = file_path
        self.larva_number = larva_number
        self.study_type = study_type
        self.dataset = dataset
        self.x_positions = []
        self.y_positions = []
        self.x_positions_mm = []
        self.y_positions_mm = []
        # Tune the scale here
        self.scale_mm_to_pixel = 0.2  # mm/px

    def read_positions_from_csv(self) -> None:
        """Read positions from the CSV file."""
        with open(self.file_path, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                self.x_positions.append(float(row['X']))
                self.y_positions.append(float(row['Y']))

    def scale_positions_to_mm(self) -> None:
        """Convert coordinates to millimeters."""
        self.x_positions_mm = [round(x * self.scale_mm_to_pixel, 2) for x in self.x_positions]
        self.y_positions_mm = [round(y * self.scale_mm_to_pixel, 2) for y in self.y_positions]

    def plot_trajectory_px(self) -> None:
        """Plot the trajectory in pixels."""
        start_x_px = self.x_positions[0]
        start_y_px = self.y_positions[0]
        end_x_px = self.x_positions[-1]
        end_y_px = self.y_positions[-1]

        plt.figure(figsize=(6, 6))
        plt.plot(self.x_positions, self.y_positions, marker='o', linestyle='-', color='k', markersize=3, zorder=1)
        plt.scatter(start_x_px, start_y_px, color='r', s=400, label='Start', zorder=2)
        plt.scatter(end_x_px, end_y_px, color='g', s=400, label='End', zorder=2)

        plt.gca().invert_yaxis()
        plt.title(f'Movement trajectory of larva {self.larva_number} (pixels)', fontsize=18)
        plt.xlabel('X position (pixels)')
        plt.ylabel('Y position (pixels)')
        plt.grid(True)
        plt.legend(loc='upper left')

        self.save_plot(f'plots/{self.dataset}/{self.study_type}/larva_{self.larva_number}_trajectory_px.png')
        plt.show()

    def plot_trajectory_mm(self) -> None:
        """Plot the trajectory in millimeters."""
        start_x_mm = self.x_positions_mm[0]
        start_y_mm = self.y_positions_mm[0]
        end_x_mm = self.x_positions_mm[-1]
        end_y_mm = self.y_positions_mm[-1]

        plt.figure(figsize=(8, 8))
        plt.plot(self.x_positions_mm, self.y_positions_mm, marker='o', linestyle='-', color='b', markersize=3, zorder=1)
        plt.scatter(start_x_mm, start_y_mm, color='r', s=400, label='Start', zorder=2)
        plt.scatter(end_x_mm, end_y_mm, color='g', s=400, label='End', zorder=2)

        plt.gca().invert_yaxis()
        plt.title(f'Movement trajectory of larva {self.larva_number} (mm)', fontsize=18)
        plt.xlabel('X position (mm)')
        plt.ylabel('Y position (mm)')
        plt.grid(True)
        plt.legend(loc='upper left')

        self.save_plot(f'plots/{self.dataset}/{self.study_type}/larva_{self.larva_number}_trajectory_mm.png')
        plt.show()

    def save_plot(self, filename: str) -> None:
        """Ensure the folder exists and save the plot to a file."""
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        plt.savefig(filename)

    def run(self) -> None:
        """Run the load, scale and plot procedure."""
        self.read_positions_from_csv()
        self.scale_positions_to_mm()
        self.plot_trajectory_px()
        self.plot_trajectory_mm()


if __name__ == "__main__":
    dataset = get_dataset()
    study_type = get_study_type()
    larva_number = get_larva_number()

    try:
        file_path = f'tracking/{dataset}/{study_type}/larva_positions_{larva_number}.csv'
        if not os.path.isfile(file_path):
            print(f"No trajectory file for this larva: {file_path}")
        else:
            plotter = LarvaeTrajectoryPlotter(file_path, larva_number, study_type, dataset)
            plotter.run()
    except ValueError:
        print("Invalid value. Make sure you enter an integer. Try again.")

## 5. Reverse engineering (path from angles & distances)

In [ ]:
class LarvaReverseTrajectory:
    def __init__(self, file_path: str, larva_number: int, study_type: str, dataset: str, scaling_factor: float = 0.2):  # tune the scale
        self.file_path = file_path
        self.study_type = study_type
        self.dataset = dataset
        self.larva_number = larva_number
        self.scaling_factor = scaling_factor
        self.df = None

    def load_and_scale_data(self):
        """Load data from the CSV file and scale the coordinates."""
        self.df = pd.read_csv(self.file_path)
        self.df['X'] = (self.df['X'] * self.scaling_factor).round(2)
        self.df['Y'] = (self.df['Y'] * self.scaling_factor).round(2)

    def calculate_angles_and_distances(self):
        """Compute angles and distances from coordinate differences."""
        dx = self.df["X"].diff()
        dy = self.df["Y"].diff()
        angles = np.degrees(np.arctan2(-dx, dy))
        angles_clockwise = (360 - angles) % 360
        distance = np.sqrt(dx**2 + dy**2)

        self.df["Angle"] = angles_clockwise.round().astype("Int64")
        self.df["Distance"] = distance.round(2)
        self.df.dropna(subset=["Angle", "Distance"], inplace=True)

    def save_dataframe(self, study_type):
        """Save the DataFrame to a CSV file."""
        # Create tracking/<dataset>/<study_type> if it does not exist.
        output_dir = Path('tracking') / str(self.dataset) / str(study_type)
        output_dir.mkdir(parents=True, exist_ok=True)

        output_file = output_dir / f'larva_positions_{self.larva_number}_with_angles.csv'
        self.df.to_csv(output_file, index=False)

    def calculate_path(self):
        """Reconstruct the path from angles and distances."""
        x_pos, y_pos = self.df.iloc[0]['X'], self.df.iloc[0]['Y']
        x_path, y_path = [x_pos], [y_pos]

        for _, row in self.df.iterrows():
            angle_rad = np.radians(row['Angle'])
            delta_x = row['Distance'] * np.sin(angle_rad)
            delta_y = row['Distance'] * np.cos(angle_rad)
            x_pos += delta_x
            y_pos += delta_y
            x_path.append(x_pos)
            y_path.append(y_pos)

        return x_path, y_path

    def plot_trajectory(self, x_path: list, y_path: list, save: bool = True):
        """Plot the reconstructed path and optionally save it."""
        start_x, start_y = x_path[0], y_path[0]
        end_x, end_y = x_path[-1], y_path[-1]

        plt.figure(figsize=(8, 8))
        plt.plot(x_path, y_path, marker='o', linestyle='-', color='b', markersize=3)
        plt.scatter(start_x, start_y, color='r', s=200, label='Start', zorder=2)
        plt.scatter(end_x, end_y, color='g', s=200, label='End', zorder=2)
        plt.title(f'Larva {self.larva_number} - path reconstructed from angles & distances', fontsize=18)
        plt.xlabel('X position')
        plt.ylabel('Y position')
        plt.grid(True)
        plt.legend(loc='upper left')
        plt.gca().invert_yaxis()

        if save:
            plot_file = f'plots/{self.dataset}/{self.study_type}/larva_{self.larva_number}_reverse_trajectory.png'
            os.makedirs(os.path.dirname(plot_file), exist_ok=True)
            plt.savefig(plot_file)

        plt.show()

    def run(self, study_type):
        """Run the full processing and plotting step by step."""
        self.load_and_scale_data()
        self.calculate_angles_and_distances()
        self.save_dataframe(study_type)

        x_path, y_path = self.calculate_path()
        self.plot_trajectory(x_path, y_path)


if __name__ == "__main__":
    dataset = get_dataset()
    study_type = get_study_type()
    larva_number = get_larva_number()

    try:
        file_path = f'tracking/{dataset}/{study_type}/larva_positions_{larva_number}.csv'
        if not os.path.isfile(file_path):
            print(f"No trajectory file for this larva: {file_path}")
        else:
            plotter = LarvaReverseTrajectory(file_path, larva_number, study_type, dataset)
            plotter.run(study_type)
    except ValueError:
        print("Invalid value. Make sure you enter an integer. Try again.")

## 6. Heatmap (most visited positions)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

class LarvaHeatmap:
    def __init__(self, file_path: str, larva_num: int, study_type: str, dataset: str, scale: float = 0.2, plate_size_mm: int = 95):  # tune scale and map size
        self.file_path = file_path
        self.larva_num = larva_num
        self.study_type = study_type
        self.dataset = dataset
        self.scale = scale
        self.plate_size_mm = plate_size_mm
        self.df = None
        self.filtered_positions = None
        self.unique_visited_pixels = None
        self.percentage_coverage = 0

    def load_data(self):
        """Load the CSV file."""
        self.df = pd.read_csv(self.file_path)

    def process_data(self):
        """Process data to extract unique visited pixels."""
        grouped = self.df.groupby(['X', 'Y', 'Angle']).size().reset_index(name='count')
        self.filtered_positions = grouped[['X', 'Y']]
        self.unique_visited_pixels = self.filtered_positions.drop_duplicates()

    def calculate_coverage(self):
        """Compute the percentage of visited pixels."""
        plate_size_px = int(self.plate_size_mm / self.scale)
        total_available_pixels = plate_size_px * plate_size_px
        visited_pixel_count = self.unique_visited_pixels.shape[0]
        self.percentage_coverage = (visited_pixel_count / total_available_pixels) * 100
        display(HTML(f"<h3><span style='color: red;'>Visited pixels: {self.percentage_coverage:.2f}%"))

    def plot_heatmap(self, save: bool = True):
        """Create and optionally save the heatmap."""
        output_file = f'plots/{self.dataset}/{self.study_type}/larva_{self.larva_num}_heatmap_trace.png'
        os.makedirs(os.path.dirname(output_file), exist_ok=True)

        plt.figure(figsize=(10, 8))
        plt.hist2d(
            self.filtered_positions["X"],
            self.filtered_positions["Y"],
            bins=100,  # set this to emulate a Trace for deep learning
            cmap="hot",
            density=False,
        )

        plt.axis('off')
        plt.gca().invert_yaxis()

        if save:
            plt.savefig(output_file, format='png', dpi=300)

        plt.show()

    def run(self):
        """Run all analysis and visualization steps."""
        self.load_data()
        self.process_data()
        self.calculate_coverage()
        self.plot_heatmap()


if __name__ == "__main__":
    dataset = get_dataset()
    study_type = get_study_type()
    larva_number = get_larva_number()

    try:
        file_path = f'tracking/{dataset}/{study_type}/larva_positions_{larva_number}_with_angles.csv'
        if not os.path.isfile(file_path):
            print(f"No trajectory file for this larva: {file_path}")
        else:
            heatmap_creator = LarvaHeatmap(file_path, larva_num=larva_number, study_type=study_type, dataset=dataset)
            heatmap_creator.run()
    except ValueError:
        print("Invalid value. Make sure you enter an integer. Try again.")

## 7. Bubble plot (most visited positions)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

class LarvaBubblePlotter:
    def __init__(self, file_path: str, larva_num: int, study_type: str, dataset: str, n: int):
        self.file_path = file_path
        self.larva_num = larva_num
        self.study_type = study_type
        self.dataset = dataset
        self.n = n
        self.df = None
        self.grouped = None
        self.top_n_locations = None
        self.top_10_locations = None
        self.plots_dir = f'plots/{dataset}/{study_type}'
        self.validate_inputs()

    def validate_inputs(self):
        """Validate the inputs."""
        if not os.path.isfile(self.file_path):
            raise FileNotFoundError(f"File does not exist: {self.file_path}")
        if not isinstance(self.n, int) or self.n <= 0:
            raise ValueError("'n' must be a positive integer.")

    def load_and_process_data(self):
        """Load and process the data."""
        self.df = pd.read_csv(self.file_path)
        self.df['X'] = self.df['X'].round(2)
        self.df['Y'] = self.df['Y'].round(2)
        self.grouped = self.df.groupby(['X', 'Y']).size().reset_index(name='count')
        self.top_n_locations = self.grouped.nlargest(self.n, 'count')
        self.top_10_locations = self.grouped.nlargest(10, 'count')

    def plot_bubble_chart(self):
        """Generate and save the bubble chart."""
        os.makedirs(self.plots_dir, exist_ok=True)

        x_top_n = self.top_n_locations['X']
        y_top_n = self.top_n_locations['Y']
        counts_top_n = self.top_n_locations['count']

        plt.figure(figsize=(8, 8))
        plt.scatter(x_top_n, y_top_n, s=counts_top_n*10, c='blue', alpha=0.5, edgecolors='k')
        plt.title(f"Larva {self.larva_num} - {self.n} most visited locations", fontdict={'fontsize': 18})
        plt.xlabel("X position")
        plt.ylabel("Y position")
        plt.gca().invert_yaxis()
        plt.grid(True)
        plt.savefig(f'{self.plots_dir}/larva_{self.larva_num}_top_{self.n}_most_visited_locations.png')
        plt.show()

    def create_table(self):
        """Generate and save the table of most visited locations."""
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.axis('tight')
        ax.axis('off')

        table_data = [['X', 'Y', 'Count']] + list(zip(self.top_10_locations['X'], self.top_10_locations['Y'], self.top_10_locations['count']))
        cell_colors = [['lightgrey', 'lightgrey', 'lightgrey']] + [['lavender', 'lavender', 'lavender'] for _ in range(len(self.top_10_locations))]

        table = ax.table(cellText=table_data, loc='center', cellColours=cell_colors)

        for (i, j), cell in table.get_celld().items():
            if i == 0:
                cell.set_text_props(fontweight='bold')

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 1.5)

        plt.title(f"Top 10 most visited locations - Larva {self.larva_num}", fontsize=16)
        plt.savefig(f'{self.plots_dir}/larva_{self.larva_num}_top_10_most_visited_locations_table.png')
        plt.show()

    def run(self):
        """Run all analysis steps."""
        self.load_and_process_data()
        self.plot_bubble_chart()
        self.create_table()


if __name__ == "__main__":
    dataset = get_dataset()
    study_type = get_study_type()
    larva_number = get_larva_number()

    try:
        file_path = f"tracking/{dataset}/{study_type}/larva_positions_{larva_number}_with_angles.csv"

        if not os.path.isfile(file_path):
            print(f"No trajectory file for this larva: {file_path}")
        else:
            samples_input = input("Enter the number of positions to display: ")
            try:
                samples_count = int(samples_input)
                larva_analysis = LarvaBubblePlotter(file_path, larva_num=larva_number, study_type=study_type, dataset=dataset, n=samples_count)
                larva_analysis.run()
            except ValueError:
                print("Invalid number of positions. Make sure you enter an integer.")
    except ValueError:
        print("Invalid larva number. Make sure you enter an integer.")

## 8. Rose diagram (movement directions)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

class LarvaAngleAnalysis:
    def __init__(self, file_path: str, larva_num: int, study_type: str, dataset: str, bin_size: int = 15):  # tune the number of angle bins
        self.file_path = file_path
        self.larva_num = larva_num
        self.study_type = study_type
        self.dataset = dataset
        self.bin_size = bin_size
        self.df = None
        self.filtered_grouped = None
        self.total_occurrences = 0
        self.direction_df = None
        self.plots_dir = f'plots/{dataset}/{study_type}'
        self.validate_inputs()

    def validate_inputs(self):
        """Validate the inputs."""
        if not os.path.isfile(self.file_path):
            raise FileNotFoundError(f"File does not exist: {self.file_path}")
        if not isinstance(self.bin_size, int) or self.bin_size <= 0:
            raise ValueError("bin_size must be a positive integer.")

    def load_data(self):
        """Load the CSV file."""
        self.df = pd.read_csv(self.file_path)

    def process_data(self):
        """Process data to extract unique angles."""
        grouped = self.df.groupby(['X', 'Y', 'Angle']).size().reset_index(name='count')
        self.filtered_grouped = grouped.groupby(['X', 'Y']).filter(lambda x: len(x['Angle'].unique()) == 1)
        self.total_occurrences = self.filtered_grouped['count'].sum()

    def plot_rose_diagram(self):
        """Generate the polar (rose) diagram."""
        angles = self.filtered_grouped["Angle"]
        angles_radians = np.deg2rad(angles)
        n_bins = int(360 / self.bin_size)

        plt.figure(figsize=(8, 8))
        ax = plt.subplot(111, polar=True)

        counts, bins = np.histogram(angles_radians, bins=np.linspace(-np.deg2rad(self.bin_size)/2,
                                                                   2*np.pi-np.deg2rad(self.bin_size)/2,
                                                                   n_bins + 1))
        ax.bar(bins[:-1], counts, width=np.deg2rad(self.bin_size), edgecolor='k', align='edge')
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)

        plt.title(f"Larva {self.larva_num} - angle distribution (rose diagram)", pad=30, fontdict={'fontsize': 16})
        os.makedirs(self.plots_dir, exist_ok=True)
        plt.savefig(f'{self.plots_dir}/larva_{self.larva_num}_angle_rose_diagram.png')
        plt.show()

        display(HTML(f"<h3><span style='color: red;'>Total occurrences of angles: {self.total_occurrences}</h3>"))

    def create_direction_table(self):
        """Create the angle table."""
        n_bins = int(360 / self.bin_size)
        direction_labels = [f"{i * self.bin_size}-{(i + 1) * self.bin_size} deg" for i in range(n_bins)]

        angles_radians = np.deg2rad(self.filtered_grouped["Angle"])
        counts, _ = np.histogram(angles_radians, bins=np.linspace(-np.deg2rad(self.bin_size)/2,
                                                                   2*np.pi-np.deg2rad(self.bin_size)/2,
                                                                   n_bins + 1))

        direction_counts = {label: 0 for label in direction_labels}
        for i in range(len(counts)):
            direction_counts[direction_labels[i]] += counts[i]

        self.direction_df = pd.DataFrame.from_dict(direction_counts, orient='index', columns=['Occurrences']).reset_index()
        self.direction_df = self.direction_df.rename(columns={'index': 'Direction'})
        self.direction_df['Percentage'] = (self.direction_df['Occurrences'] / self.total_occurrences) * 100
        self.direction_df['Percentage'] = self.direction_df['Percentage'].round(2)

        summary_row = pd.DataFrame({'Direction': ['All'], 'Occurrences': [self.total_occurrences], 'Percentage': [100.0]})
        self.direction_df = pd.concat([self.direction_df, summary_row], ignore_index=True)
        self.direction_df = self.direction_df[:-1].sort_values(by='Occurrences', ascending=False)
        self.direction_df = pd.concat([self.direction_df, summary_row], ignore_index=True)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.axis('tight')
        ax.axis('off')

        table_data = [['Direction', 'Occurrences', 'Percentage']] + list(zip(self.direction_df['Direction'], self.direction_df['Occurrences'], self.direction_df['Percentage']))
        cell_colors = [['lightgrey', 'lightgrey', 'lightgrey']] + [['lavender', 'lavender', 'lavender'] for _ in range(len(self.direction_df)-1)] + [['lightblue', 'lightblue', 'lightblue']]

        table = ax.table(cellText=table_data, loc='center', cellColours=cell_colors)
        for (i, j), cell in table.get_celld().items():
            if i == 0:
                cell.set_text_props(fontweight='bold')
            if j == 1 and i == len(table_data) - 1:
                cell.set_text_props(fontweight='bold', color='black')

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 1.3)

        plt.savefig(f'{self.plots_dir}/larva_{self.larva_num}_angle_table.png')
        plt.show()

    def plot_direction_bar_chart(self):
        """Generate the bar chart."""
        plt.figure(figsize=(8, 8))
        plt.bar(self.direction_df['Direction'][:-1], self.direction_df['Percentage'][:-1], color='skyblue')
        plt.xlabel('Directions')
        plt.ylabel('Percentage [%]')
        plt.title(f'Larva {self.larva_num} - angle distribution (%)')
        plt.xticks(rotation=45)
        plt.grid(axis='y')
        plt.savefig(f'{self.plots_dir}/larva_{self.larva_num}_angle_plot.png')
        plt.show()

    def run(self):
        """Run the full analysis."""
        self.load_data()
        self.process_data()
        self.plot_rose_diagram()
        self.create_direction_table()
        self.plot_direction_bar_chart()


if __name__ == "__main__":
    dataset = get_dataset()
    study_type = get_study_type()
    larva_number = get_larva_number()

    try:
        file_path = f"tracking/{dataset}/{study_type}/larva_positions_{larva_number}_with_angles.csv"

        if not os.path.isfile(file_path):
            print(f"No trajectory file for this larva: {file_path}")
        else:
            analysis = LarvaAngleAnalysis(file_path, larva_num=larva_number, study_type=study_type, dataset=dataset)
            analysis.run()
    except ValueError:
        print("Invalid larva number. Make sure you enter an integer.")